<!-- FILE MAP | Self-contained Colab runner for the ensemble / weighted-ensemble / box-cascade
     experiment (docs/PLAN_ENSEMBLE.md). Step 0 restores the Drive results mirror — results/ is
     gitignored, so a fresh clone has none of it, and verification + the cost table both read
     published rows. Cells then run Phases 0-4 end to end in one session: the 2 GB probability
     cache lives on session disk and must not outlive the session. -->

# Ensembles + box cascade (Colab)

**Edit only cell 1** (which ensemble/cascade specs and seeds to run), then *Runtime -> Run all*.

- Needs the nine trained checkpoints (`unet`/`sam_vit_h`/`sam_vit_b` x seed 42/43/44) and the
  three base backbone `.pth` files. The Phase 0 gate (cell 6) checks all of it before anything
  else runs.
- Every ensemble/cascade row reads only cached per-image probabilities — no ground truth at
  inference — and mirrors its `metrics.json` + `inference.json` to Drive as it goes.

In [ ]:
# 1. Pick what to run — the ONLY cell you should need to edit. [TWEAK]
ENSEMBLE_SPECS_TO_RUN = ['ens_unet_samh', 'ens_top3', 'ens_unet_samh_fitted', 'ens_top3_fitted']
# ens_unet_samh(_fitted) = U-Net + SAM-ViT-H (member set A); ens_top3(_fitted) = + SAM-ViT-B (set B)
CASCADE_SPECS_TO_RUN = ['oracle_casc_gtbox_medsam', 'casc_unet_medsam']
# oracle_casc_gtbox_medsam is the GT-box ceiling gate — it always runs first, seed0 only.
SEEDS = [42, 43, 44]   # ensembles + the fair cascade row are seed-matched to the trained runs
CACHE_DIR = 'cache'    # [TWEAK] point at a Drive path to survive a session restart

In [ ]:
# 2. Mount Google Drive (results mirror, checkpoints, and the sidecar mirror all live here).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. Environment: pull latest code, install deps, fetch the base backbones.
import os, sys, subprocess
from pathlib import Path

REPO = '/content/msu2026summer_final_project'
if not os.path.exists(REPO):
    !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
%cd {REPO}
BRANCH = 'worktree-ensemble-plan'  # [TWEAK] branch that carries the ensemble code
!git fetch --quiet origin && git checkout --quiet {BRANCH} && git reset --hard origin/{BRANCH}
!find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from src.config import load_run_config
cfg = load_run_config('configs/run.yaml')
print('Environment ready.')

In [ ]:
# 4. Step 0 (mandatory, first): restore the Drive results mirror into ./results/.
#    results/ is gitignored -- a fresh clone has none of it, and every check below (the Phase 0
#    gate, predict_cache.py's verification, the cost table) reads a published metrics.json.
import shutil

drive_results = Path(cfg['output']['drive_results_dir'])
local_results = Path('results')
if drive_results.exists():
    shutil.copytree(drive_results, local_results, dirs_exist_ok=True)
n_restored = len(list(local_results.glob('*/seed*/metrics.json')))
print(f'Restored {n_restored} metrics.json file(s) from {drive_results}.')
assert n_restored >= 18, (
    f'Only {n_restored} published metrics.json restored (expected >= 18: 9 trained runs x 2, '
    f'plus oracle rows). Check that Drive is mounted and msu2026_checkpoints/results/ exists '
    f'before continuing -- every later step needs the published rows.'
)

In [ ]:
# 5. Phase 0 gate: every base backbone, the segment_anything package, the data splits, the
#    nine trained checkpoints, and the nine published metrics.json rows, all in one table.
!python predict_cache.py --config configs/run.yaml --inventory

In [ ]:
# 6. Phase 1: build the per-image probability cache for every (model, seed) the ensembles
#    and the cascade need. Verifies each cache against its published metrics.json (tolerance
#    1e-3) before trusting it, then writes + mirrors the inference.json cost sidecar.
MEMBERS = ['unet', 'sam_lora', 'sam_b']
for model in MEMBERS:
    for seed in SEEDS:
        args = [sys.executable, 'predict_cache.py', '--config', 'configs/run.yaml',
               '--model', model, '--seed', str(seed), '--cache-dir', CACHE_DIR]
        print('=' * 70); print(f'>>> predict_cache: {model} seed{seed}'); print('=' * 70, flush=True)
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
        if proc.wait() != 0:
            raise RuntimeError(
                f'predict_cache.py failed for {model} seed{seed} (see the printed verification table above). See docs/PLAN_ENSEMBLE.md Phase 1 for the --accept-drift override.'
            )

In [ ]:
# 7. Phase 2: self-check every member's cache, then the uniform ensembles.
for model in MEMBERS:
    !python ensemble_eval.py --config configs/run.yaml --self-check --member {model} --seed 42 --cache-dir {CACHE_DIR}

for spec in [s for s in ENSEMBLE_SPECS_TO_RUN if not s.endswith('_fitted')]:
    for seed in SEEDS:
        !python ensemble_eval.py --config configs/run.yaml --spec {spec} --seed {seed} --cache-dir {CACHE_DIR}

In [ ]:
# 8. Phase 3: fitted ensembles (weights fit on the first contiguous half of each seen split).
for spec in [s for s in ENSEMBLE_SPECS_TO_RUN if s.endswith('_fitted')]:
    for seed in SEEDS:
        !python ensemble_eval.py --config configs/run.yaml --spec {spec} --seed {seed} --cache-dir {CACHE_DIR}

In [ ]:
# 9. Phase 4: box cascade. The GT-box ceiling gate runs first (seed0 only, must land within
#    0.02 of the published 0.9246 unseen mDice) before the fair, detector-driven row.
if 'oracle_casc_gtbox_medsam' in CASCADE_SPECS_TO_RUN:
    !python cascade_eval.py --config configs/run.yaml --spec oracle_casc_gtbox_medsam

if 'casc_unet_medsam' in CASCADE_SPECS_TO_RUN:
    for seed in SEEDS:
        !python cascade_eval.py --config configs/run.yaml --spec casc_unet_medsam --seed {seed} --cache-dir {CACHE_DIR}

In [ ]:
# 10. Regenerate the summary (derived table + cost table) from everything written above.
!python aggregate_results.py
print('\nNext: read results/summary/SUMMARY.md, then update docs/FINDINGS.md by hand '
     '(docs/PLAN_ENSEMBLE.md Phase 5).')